# Lezione 6 — NumPy: pensare per array

**Come si usa questo notebook.** Come sempre: leggi, esegui, prova
l'esercizio prima della soluzione, chiudi con il progetto. Da qui comincia la
**Fase 0**: i fondamenti numerici che la Fase 2 (reti neurali) darà per
scontati.

**Cosa saprai fare alla fine:** ragionare "per array" invece che "per cicli":
è il modo in cui funziona tutto il calcolo numerico moderno — NumPy, pandas,
TensorFlow, PyTorch condividono esattamente questo modello mentale.

## Parte 1 — Teoria: perché esiste NumPy

Python è lento sui numeri. Non per cattiva progettazione: ogni numero Python
è un oggetto completo, e un ciclo `for` che somma un milione di elementi paga
il costo dell'interprete a ogni passo.

NumPy risolve il problema con una sola idea: **l'array** — un blocco di
memoria contiguo di numeri tutti dello stesso tipo, su cui le operazioni
girano in C, in blocco. Da questa idea discende il principio operativo che
userai ogni giorno:

> **Vettorizzare**: esprimere l'operazione su *tutto l'array in una volta*,
> non elemento per elemento.

`dati * 2` moltiplica un milione di numeri senza un solo ciclo Python
visibile. Non è solo più veloce (10-100x): è anche più leggibile, perché il
codice dice *cosa* calcoli, non *come* iterare.

Questo è il concetto trasferibile della lezione: quando in TensorFlow
scriverai `y = W @ x + b`, starai facendo esattamente questo — operazioni su
interi blocchi di numeri, mai cicli. Chi pensa per cicli combatte contro il
framework; chi pensa per array lo usa.

In [ ]:
import numpy as np
import time

dati = list(range(1_000_000))
array = np.arange(1_000_000)

t0 = time.perf_counter()
quadrati_lenti = [x * x for x in dati]           # ciclo Python
t1 = time.perf_counter()
quadrati_veloci = array * array                   # vettorizzato
t2 = time.perf_counter()

print(f'Ciclo Python : {(t1 - t0) * 1000:6.1f} ms')
print(f'Vettorizzato : {(t2 - t1) * 1000:6.1f} ms')

## Parte 2 — Teoria: gli strumenti quotidiani

Quattro idiomi coprono la maggior parte del lavoro reale:

1. **Operazioni elemento per elemento**: `a + b`, `a * 2`, `np.sqrt(a)` —
   si applicano a ogni elemento, restituiscono un array della stessa forma.
2. **Riduzioni**: `a.sum()`, `a.mean()`, `a.max()` — da molti numeri a uno.
   Con `axis=` riduci solo lungo una dimensione (fondamentale per le tabelle:
   `axis=0` = per colonna, `axis=1` = per riga).
3. **Maschere booleane**: `a[a > 0]` — un confronto produce un array di
   True/False che seleziona elementi. È l'equivalente NumPy dei filtri pandas
   che usi dalle Lezioni 1-2 (pandas è costruito sopra NumPy).
4. **Slicing**: `a[10:20]`, `tabella[:, 2]` (tutte le righe, colonna 2) —
   viste sui dati senza copiarli.

Vediamoli lavorare insieme su un caso familiare — le temperature:

In [ ]:
rng = np.random.default_rng(0)
temperature = rng.normal(18, 5, 1000)

sopra_media = temperature[temperature > temperature.mean()]
print(f'media {temperature.mean():.1f} gradi; {len(sopra_media)} letture sopra la media')

tabella = rng.normal(18, 5, (4, 3))    # 4 letture x 3 stazioni
print('\nmedia per stazione (axis=0):', tabella.mean(axis=0).round(1))
print('media per lettura  (axis=1):', tabella.mean(axis=1).round(1))

## Parte 3 — Esercizio guidato

Questo ciclo calcola lo **z-score** di ogni temperatura (quante deviazioni
standard dista dalla media — lo standardizing della Lezione 5, riconoscilo):

```python
z = []
for t in temperature:
    z.append((t - temperature.mean()) / temperature.std())
```

Funziona, ma ricalcola media e deviazione a ogni giro ed è puro stile
"ciclo". Il tuo compito: riscrivilo **vettorizzato** (una riga), verifica che
il risultato sia identico con `np.allclose`, e conta con una maschera
booleana quante letture hanno |z| > 2 (i candidati outlier statistici della
Lezione 2, rivisti in chiave NumPy).

**Prova tu:**

In [ ]:
# Scrivi qui il tuo tentativo:
# 1) z vettorizzato in una riga
# 2) confronto con la versione a ciclo (np.allclose)
# 3) quanti |z| > 2?

temperature[:5]

### Soluzione spiegata

- media e deviazione si calcolano **una volta**, fuori dall'espressione;
- l'espressione `(temperature - media) / dev` è l'intera operazione su 1000
  elementi: NumPy sottrae e divide in blocco;
- `np.abs(z) > 2` è un array di booleani; la sua `sum()` conta i True —
  l'idioma "conta quanti soddisfano la condizione" che userai ovunque.

In [ ]:
media, dev = temperature.mean(), temperature.std()
z = (temperature - media) / dev

z_lento = []
for t in temperature:
    z_lento.append((t - media) / dev)
assert np.allclose(z, np.array(z_lento))

candidati = int((np.abs(z) > 2).sum())
print(f'{candidati} letture su {len(z)} hanno |z| > 2')

## Parte 4 — Il progetto: Memory AI Lab, passo 6

La matrice di feature della Lezione 5 finora l'hai vista come tabella pandas.
Da oggi la guardi come la vedrà il modello: **un array di numeri**. Il passo
di oggi è una domanda che decide tutto il seguito: *le feature contengono
segnale sui tipi di memoria?* Se le medie delle feature differiscono tra le
classi, un classificatore ha qualcosa da imparare.

In [ ]:
import pandas as pd
from pathlib import Path

train = pd.read_csv(Path('..') / 'datasets' / 'processed' / 'memory_features_train.csv')
X = train.drop(columns='target').to_numpy()
y = train['target'].to_numpy()
print(f'X: {X.shape} (esempi x feature)   y: {y.shape}   dtype: {X.dtype}')

In [ ]:
classi = ['episodic', 'preference', 'semantic', 'unknown']
nomi_feature = train.columns[:-1]

print('Media delle feature per classe (feature standardizzate):\n')
intestazione = 'classe      ' + ''.join(f'{nome[:12]:>14}' for nome in nomi_feature[:3])
print(intestazione)
for k, nome in enumerate(classi):
    maschera = y == k                       # maschera booleana: la Parte 2 al lavoro
    medie = X[maschera][:, :3].mean(axis=0)  # slicing + riduzione per colonna
    print(f'{nome:<12}' + ''.join(f'{m:14.2f}' for m in medie))

Leggi la tabella: se le classi hanno medie diverse su `text_length` o
`word_count`, c'è segnale — le memorie di tipo diverso *sono scritte
diversamente*, e un modello potrà sfruttarlo. Hai appena fatto la tua prima
analisi "da modello": maschere, slicing e riduzioni su un array.

## Cosa hai imparato

- NumPy esiste per un motivo: array contigui + operazioni in blocco.
- **Vettorizzare** = esprimere l'operazione sull'intero array; i cicli sui
  numeri sono un segnale d'allarme.
- I quattro idiomi: elemento-per-elemento, riduzioni (con `axis`), maschere
  booleane, slicing.
- pandas, TensorFlow e PyTorch parlano tutti questo linguaggio.

## Quiz

1. `tabella.mean(axis=0)` su una tabella (100 righe x 5 colonne): che forma
   ha il risultato e cosa rappresenta?
2. Perché la versione vettorizzata dello z-score è più veloce di 10-100x?
3. Cosa restituisce `(np.abs(z) > 2).sum()` e perché funziona?

<details>
<summary><b>Apri le risposte</b></summary>

1. Forma `(5,)`: una media per colonna. `axis=0` riduce lungo le righe,
   "schiacciando" la tabella verticalmente.
2. Perché l'intera operazione avviene in C su memoria contigua, senza il
   costo dell'interprete Python a ogni elemento — e senza ricalcolare
   media/deviazione a ogni giro.
3. Il numero di elementi con |z| > 2. Il confronto produce booleani, e la
   somma li tratta come 1/0: l'idioma standard per contare condizioni.
</details>

## Fonti

- NumPy, *the absolute basics for beginners*:
  https://numpy.org/doc/stable/user/absolute_beginners.html
- NumPy, *Array programming with NumPy* (perché la vettorizzazione):
  https://numpy.org/doc/stable/user/whatisnumpy.html

## Prossima lezione

Gli array hanno **forme**, e le forme sono il linguaggio dei modelli:
vettori, matrici, tensori e la moltiplicazione che li fa parlare. Lezione 7.